# 03 — Silver → Gold Business Table

## Student Demo

We will create one simple business table:

```text
Daily Sales Summary

sales_date
order_count
units_sold
total_sales
average_order_value
```


In [ ]:
from pyspark.sql import functions as F

CATALOG = "retail_catalog"

orders = spark.table(f"{CATALOG}.silver.orders")
products = spark.table(f"{CATALOG}.silver.products")

display(orders.limit(10))


## 1. Create business-ready sales data


In [ ]:
gold_sales = (
    orders
    .groupBy("order_date")
    .agg(
        F.countDistinct("order_id").alias("order_count"),
        F.sum("quantity").alias("units_sold"),
        F.sum("sales_amount").alias("total_sales"),
        F.avg("sales_amount").alias("average_order_value")
    )
    .withColumnRenamed("order_date", "sales_date")
    .orderBy("sales_date")
)

display(gold_sales)


## 2. Save Gold table


In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold")

(
    gold_sales.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOG}.gold.daily_sales")
)

print("Gold business table created")


## 3. Business query


In [ ]:
%sql
SELECT
    sales_date,
    order_count,
    units_sold,
    ROUND(total_sales, 2) AS total_sales,
    ROUND(average_order_value, 2) AS average_order_value
FROM retail_catalog.gold.daily_sales
ORDER BY sales_date;


## Student explanation

> Gold is designed for business users. Instead of exposing raw transaction-level complexity, we provide a simple daily sales table that can directly feed Power BI.
